# Story Parser

## plan
* take json input, including a story text
* summarize the story scene in markdown
    * story description/summary, genre and scene mood
    * describe the setting, time of day, type of place
    * create a list of characters, their detailed appearance, clothing
* break the story into chunks (paragraph or dialog section)
    * for each chunk, create an image
    * create a sound file


In [1]:
import os
import shutil
import requests
from requests.exceptions import ConnectionError, Timeout, RequestException
import gradio as gr
from typing import List
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
import datetime
import re
import json
from pydantic import BaseModel
from ipyfilechooser import FileChooser
import unicodedata

from PIL import Image
from io import BytesIO
import base64


In [2]:
outputDirMSI="G:\\output\\pythonSD\\"
outputDirDell="G:\\GenerativeAIOutput\\pythonSD"
outputDir=outputDirMSI
fc = FileChooser()
fc.default_path = outputDir
fc.title = "<b>Select a story_config.json file</b>"
fc.filter_pattern = '*.json'
display(fc)

FileChooser(path='G:\output\pythonSD', filename='', title='<b>Select a story_config.json file</b>', show_hidde…

In [158]:
def currentFormattedTime():
    return datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

In [159]:
class StoryConfig(BaseModel):
    llm: str = "gemma3_4b" #qwen3-coder:30b, gemma3:4b, gemma3:12b, gpt-oss:20b
    max_json_generation_attempts: int = 5
    max_json_fix_attempts: int = 5
    repair_llm: str = "gemma3_4b"
    image_model: str = "juggernautXL_v8Rundiffusion.safetensors [aeb7e9e689]"
    auto111_url: str = "http://127.0.0.1"
    ports: List[str] = ["7860","7861"]
    max_images: int = 20
    min_chunk_length: int = 100
    steps: int = 30
    sampler_name: str = "DPM++ 2M Karras" #Euler
    positive_prompt: str = ""
    positive_image_prompt: str = ""
    negative_prompt: str = ""
    cfg_scale: int = 7
    seed: int = -1 #-1 for random
    width: int = 1024
    height: int = 1024
    data_file: str = "story.txt"

In [160]:
# Print the selected path, filename, or both
print(fc.selected_path)
print(fc.selected_filename)
print(fc.selected)

with open(fc.selected, "r") as file:
    raw_config = json.load(file)

config = StoryConfig.model_validate(raw_config)
display(config)


G:\output\pythonSD\storiesCD\2002JanArchivedShoeShopping
story_config.json
G:\output\pythonSD\storiesCD\2002JanArchivedShoeShopping\story_config.json


StoryConfig(llm='huihui_ai/qwen3-abliterated:latest', max_json_generation_attempts=5, max_json_fix_attempts=5, repair_llm='gemma3:4b', image_model='checkpoints_nsfw\\westmixV0PhotoRealistic_westmixV0.safetensors', auto111_url='http://127.0.0.1', ports=['7860', '7861'], max_images=12, min_chunk_length=100, steps=30, sampler_name='DPM++ 2M Karras', positive_prompt='The women are beautiful and unaware that they are exposing themselves.  They intimately touch and kiss other women.  when called for by the story line, explicit nudity is expected, including fully showing the vagina. clothes should be accidentally revealing.  Short dresses and miniskirts should give a peak at a bottomless woman and part of the vagina visible or a barely there thong, especially when sitting or bending forward. pubic hair may peak around tight shorts. ', positive_image_prompt='', negative_prompt='', cfg_scale=6, seed=-1, width=1024, height=1024, data_file='story.txt')

In [161]:
def slugify(value, allow_unicode=False):
    """
    Taken from https://github.com/django/django/blob/master/django/utils/text.py
    Convert to ASCII if 'allow_unicode' is False. Convert spaces or repeated
    dashes to single dashes. Remove characters that aren't alphanumerics,
    underscores, or hyphens. Convert to lowercase. Also strip leading and
    trailing whitespace, dashes, and underscores.
    """
    value = str(value)
    if allow_unicode:
        value = unicodedata.normalize('NFKC', value)
    else:
        value = unicodedata.normalize('NFKD', value).encode('ascii', 'ignore').decode('ascii')
    value = re.sub(r'[^\w\s-]', '', value.lower())
    return re.sub(r'[-\s]+', '-', value).strip('-_')

In [162]:
def create_output_dir(path):
    display ("Creating output directory at: " + path)
    try:
            os.mkdir(path)
            print(f"Directory '{path}' created successfully.")
            return True
    except FileExistsError:
            print(f"Directory '{path}' already exists.")
            return False
    except Exception as e:
            print(f"An error occurred: {e}") 
            return None   

In [163]:
def copy_file(sourceFile, destDir):
    destFile = os.path.join(destDir, os.path.basename(sourceFile))
    try:
        shutil.copy(sourceFile, destFile)
        print(f"File '{sourceFile}' successfully copied to '{destFile}'")
        return True
    except FileNotFoundError:
        print(f"Error: Source file '{sourceFile}' not found.")
        return False
    except IsADirectoryError:
        print(f"Error: Destination '{destDir}' is a directory, not a file.")
        return False
    except Exception as e:
        print(f"An error occurred: {e}")
        return False

In [ ]:
outputModelDir= fc.selected_path + "\\" + slugify(config.llm) #currentFormattedTime()
create_output_dir(outputModelDir)

outputDirSub=outputModelDir+"\\"+slugify(config.image_model)
dir_exists = create_output_dir(outputDirSub)
index=0
while (dir_exists==False): #dir exists
    index+=1
    outputDirSub= f"{outputModelDir}\\{slugify(config.image_model)}_{index}"
    dir_exists = create_output_dir(outputDirSub)

outputDir = outputDirSub

copy_file(f"{outputModelDir}\\story_gallery.json", outputDir)

In [165]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AI
DeepSeek API Key not set (and this is optional)
Groq API Key exists and begins gsk_
Grok API Key exists and begins xai-
OpenRouter API Key exists and begins sk-


In [166]:
openai = OpenAI()

# For Gemini, DeepSeek and Groq, we can use the OpenAI python client
# Because Google and DeepSeek have endpoints compatible with OpenAI
# And OpenAI allows you to change the base_url

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
deepseek_url = "https://api.deepseek.com"
groq_url = "https://api.groq.com/openai/v1"
grok_url = "https://api.x.ai/v1"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [167]:
llama32="llama3.2"
mistralThinker="hf.co/mradermacher/MistralThinker-v1.1-i1-GGUF:Q4_K_M"
qwen3coder30b="qwen3-coder:30b"
gemma3_4b="gemma3:4b"
gemma3_12b="gemma3:12b"
gptOss20b="gpt-oss:20b"

In [168]:
MODEL=config.llm #gemma3_4b
REPAIR_MODEL=config.repair_llm #gemma3_4b



In [169]:
system_story_summarizer = f"""You are a helpful assistant that summarizes stories into concise descriptions suitable for generating images. Please focus on capturing the key visual elements, settings, characters, and moods of the story in a way that can be effectively translated into image prompts.  Analyze the text below and create Markdown with the following sections:
 1. Story description - a 3-4 sentence summary of the story including genre and scene mood
 2. Story setting: describe the setting including location, type of place, time of day
 3.  create a list of characters, with their detailed appearance, clothing and other visual details. Be specific. It is very important to describe a gender, age, hair color (or bald), hair length and style, and other distinguishing features (e.g. glasses) that should be kept consistent in each image of the story. If the character is not named, give them an appropriate name based on age, gender and location.  Create key features if they are missing from the story (e.g. infer gender or age).  For example, Edgar is a 60 year old male poet, scruffy, ruffled, haggard appearance with balding black and gray hair, and unkempt curly hair, full unkempt beard.  He wears a tweed jacket with patches, and torn brown pants, scuffed dark shoes. 
 4. Key visual elements: highlight any significant objects, colors, or themes that should be included in the image generation. 
 Please format the output in Markdown with appropriate headings for each section. """

In [170]:
system_image_prompt_instruct = """You are a helpful chatbot who generates stable diffusion image prompts based on the text from a story.  You will be given a paragraph, stanza or line from a story.  For each paragraph of the story (or stanza of a poem), generate a concise stable diffusion prompt that captures the essence of the paragraph in vivid detail.  Use descriptive language and include artistic styles or techniques where appropriate.  

You will also be given a summary of the overall story to provide context.  Use this to ensure that the prompts you generate are consistent with the story's themes, characters, and settings.
It is very important to maintain consistent character appearances and settings across all prompts.  If a character is described as having specific features (e.g. age, gender, hair color, glasses) or clothing in one paragraph, ensure those details are reflected in all subsequent prompts involving that character. 

It is very important that you output only the image prompt text without any additional commentary or formatting.  The output should be a single, clear prompt suitable for input into a stable diffusion model.
 """



In [171]:
def break_text_into_paragraphs(text):
    """
    Break text into paragraphs by splitting on blank lines.
    Handles various line endings and whitespace variations.
    
    Args:
        text (str): The text to break into paragraphs
        
    Returns:
        list: List of paragraphs (non-empty strings)
    """
    # Split on one or more blank lines (handles different line endings)
    paragraphs = re.split(r'\n\s*\n+', text.strip())
    
    # Remove any leading/trailing whitespace from each paragraph
    paragraphs = [p.strip() for p in paragraphs if p.strip()]
    
    return paragraphs

In [172]:
def chunk_paragraphs(paragraphs):
    total_length = sum(len(s) for s in paragraphs)
    chunk_length = config.min_chunk_length
    if (total_length/config.min_chunk_length) > config.max_images:
        chunk_length = total_length/config.max_images
    chunks = []
    chunk = ""
    for p in paragraphs:
        chunk += f"\n{p}"
        if (len(chunk)>=chunk_length):
            chunks.append(chunk)
            chunk = ""
    if len(chunk)>0:
        chunk += f"\n{p}"
    return chunks

In [173]:
def chat(message, relevant_system_message, history = []):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    message = message.encode("ascii", "ignore").decode('ascii') 
    
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    response = ollama.chat.completions.create(model=MODEL, messages=messages, stream=False)
    if hasattr(response, 'error'):
        print(f"API Error: {response.error}")
        return ""
    if (not hasattr(response, 'choices')):
        print(f"No choices in response to generate image prompt for {paragraph}")
        return ""

    result = response.choices[0].message.content
    

    #display(Markdown(result))
    return result

In [174]:
def paragraphToImagePrompt(paragraph, story_summary):
    message = f"""{config.positive_prompt}
    Create an image prompt for the following paragraph from the story:
    {paragraph}
    
    Here is the summary of the story to provide context:
    {story_summary}
    """
    result = chat(message, system_image_prompt_instruct)

    return result

In [175]:
def summarize_story_text_file():
    story_text = ""
    story_summary = ""
    try:
        filename = fc.selected_path + "\\" + config.data_file
        with open(filename, "r", encoding="utf8") as file:
            story_text = file.read()
            story_summary = chat(f"{config.positive_prompt} {story_text}", system_story_summarizer)
        print(f"story summary created successfully.")
    except FileNotFoundError:
        print(f"Error: The file {filename} was not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

    return story_summary, story_text

In [176]:
class StoryImage(BaseModel):
    storyline: str
    prompt: str
    imagePath: str
    def __init__(self, **data):
        super().__init__(**data)

class StoryImageList(BaseModel):
    summary: str
    paragraphs: List[StoryImage]

In [177]:
def save_json_to_file(json_string, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(json_string)

In [178]:
def Create_Story_ImagePrompts():
    story_summary, story_text = summarize_story_text_file()
    paragraphs = break_text_into_paragraphs(story_text)
    chunks = chunk_paragraphs(paragraphs)
    StoryImages = []
    index =0
    for chunk in chunks:
        index += 1
        display(f"Generating image prompt for paragraph {index}/{len(chunks)} \n{chunk}\n")
        imagePrompt = paragraphToImagePrompt(chunk, story_summary)
        StoryImages.append(StoryImage(
            storyline=chunk,
            prompt=imagePrompt,
            imagePath=f"{index:03d}.jpg"
        ))
    storyImageList = StoryImageList(
        summary =story_summary,
        paragraphs=StoryImages
    )
    

    jsonText = storyImageList.model_dump_json(indent=4)

    save_json_to_file(jsonText, outputModelDir + "/story_gallery.json")
    save_json_to_file(jsonText, outputDir + "/story_gallery.json")

    return storyImageList

In [179]:
# following https://civitai.com/articles/4090/make-a-stable-diffusion-easy-interface-with-python
# Import the libraries




In [180]:

#textToImg_url = f"http://127.0.0.1:7860/sdapi/v1/txt2img"

default_negative_prompt = "blurry, low quality, bad anatomy, lowres, error body parts, error hands and fingers, error legs and feet, error face, deformed, blurry, ugly, jpeg artifacts, ugly face, distorted face, extra limbs, mutated hands and fingers, worst quality,"

In [181]:
def find_api_port(host, ports, endpoint = "/login_check/"):
    """
    Attempts to make an API call to a specific endpoint across a list of ports.  Returns the active port or None
    """ 
    for port in ports:
        url = f"{host}:{port}{endpoint}"
        try:
            print(f"Attempting to connect to {url}...")
            # Set a timeout for the request to prevent indefinite waiting
            response = requests.get(url, timeout=10)

            # If successful, process the response and return
            if response.status_code == 200:
                print(f"Success! Connected to port {port}. Status code: {response.status_code}")
                return port
            else:
                print(f"Connected to port {port}, but received status code: {response.status_code}")
        except ConnectionError:
            print(f"Port {port} is closed or service is unreachable.")
        except Timeout:
            print(f"Connection to port {port} timed out.")
        except RequestException as e:
            print(f"An error occurred while connecting to port {port}: {e}")

    print("Failed to connect to any of the specified ports.")
    return None     

In [182]:
auto111_port = find_api_port(
    config.auto111_url, 
    config.ports,
    "/login_check/"
)


Attempting to connect to http://127.0.0.1:7860/login_check/...
Success! Connected to port 7860. Status code: 200


In [183]:
def post_json_to_api(host, port, endpoint, headers, json_data):
    """
    Attempts to make an API call to a specific host, port and endpoint.
    """

    url = f"{host}:{port}{endpoint}"
    try:
        print(f"Attempting to connect to {url}...")
        # Set a timeout for the request to prevent indefinite waiting
        response = requests.post(url, data=json_data, headers=headers)
        #response = requests.get(url, timeout=5)

        # If successful, process the response and return
        if response.status_code == 200:
            print(f"Success! Connected to port {port}. Status code: {response.status_code}")
            return response
        else:
            print(f"Connected to port {port}, but received status code: {response.status_code}")

    except ConnectionError:
        print(f"Port {port} is closed or service is unreachable.")
    except Timeout:
        print(f"Connection to port {port} timed out.")
    except RequestException as e:
        print(f"An error occurred while connecting to port {port}: {e}")

    print("Failed to connect to any of the specified ports.")
    return None

In [184]:
def set_checkpoint(model):
    headers = {'Content-Type': 'application/json'}
    payload = json.dumps({
        "sd_model_checkpoint": model
    })
    response = post_json_to_api(
        config.auto111_url,
        auto111_port,
        "/sdapi/v1/options",
        headers,
        payload)
    if (response):
        display(f"succesfully changed checkpoint to {model}")
    else:
        display("error changing checkpoint")

In [185]:
# Define the function to call the API
# Must start Automatic 1111 web server before running this code
def call_api(prompt, negative_prompt, filename):
    # Define the URL of the API endpoint
    data = {
        "prompt": prompt,
        "negative_prompt": default_negative_prompt + negative_prompt,
        "steps": config.steps, #default is 20
        "sampler_name": config.sampler_name, # DPM++ 2M Karras, #default is Euler 
        "cfg_scale": config.cfg_scale,
        "seed": config.seed, # -1 for random seed
        "width": config.width, #default is 512
        "height": config.height,
        "override_settings": {
            "sd_model_checkpoint": config.image_model #juggernautXL_v8Rundiffusion.safetensors [aeb7e9e689]
        }
    }

    # Convert the data to JSON format
    json_data = json.dumps(data)

    # Set the headers for the request
    headers = {'Content-Type': 'application/json'}

    response = post_json_to_api(
        config.auto111_url,
        auto111_port,
        "/sdapi/v1/txt2img",
        headers,
        json_data)


    # Send the POST request to the API
    #response = requests.post(textToImg_url, data=json_data, headers=headers)
    
    # Check if the request was successful (status code 200)
    if response: #.status_code == 200:
       # Decode the JSON response
        json_response = response.json()

        # Extract the base64 image data from the response
        image_data = json_response.get('images', [''])[0]

        # Decode the base64 image data
        image_bytes = base64.b64decode(image_data)

        # Open the image using PIL
        image = Image.open(BytesIO(image_bytes))

        display(f"Saving image to {filename}")

        #current_time = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        image.save(filename)  # Save the image to a file
        return image
    else:
        # Return an error message if the request was not successful
        return f"Error: {response.status_code}"



In [186]:
def generate_images_from_story(story_json):
    set_checkpoint(config.image_model)
    for story in story_json:
        prompt = f"""
        Here is the image prompt:
        {story.prompt} {config.positive_image_prompt}
         """
                
        # This is the story text that will accompany this image:
        # {story.storyline}

        storyline = story.storyline
        negative_prompt = config.negative_prompt
        image = call_api(prompt, negative_prompt,f"{outputDir}\\{story.imagePath}" )
        #display(Markdown(f"{storyline}"))
        #display(image)


In [187]:
def load_story_gallery_json(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        raw_json = json.load(file)
    out_json = StoryImageList.model_validate(raw_json)
    return out_json

In [ ]:
file_path = outputDir + "/story_gallery.json"
if os.path.exists(file_path):
    print(f"'{file_path}' exists. loading...")
    storyImages = load_story_gallery_json(file_path)
else:
    print(f"'{file_path}' does not exist. Creating json file")
    storyImages = Create_Story_ImagePrompts()

generate_images_from_story(storyImages.paragraphs)

copy_file(f".\\slideshow\\index.html", outputDir)

display("DONE...")